# 🗂️ GENI — Domain Owner: Upload & Vectorise ANZSIC Master CSV

**Run this notebook once** to upload `anzsic_master.csv` into GENI's RAG pipeline.  
GENI will parse, chunk and embed the document automatically.

At the end you will produce a `document_ids.json` file — **share this file with the operator** so they can link the document to the ANZSIC classifier bot.

### Prerequisites
- `gcloud` CLI installed and authenticated (`gcloud auth login`)
- You hold the `lumi-prod-{domain}-domainowner` role in MyAccess
- `anzsic_master.csv` is in the **same folder** as this notebook
- `requests` installed (`pip install requests`)

## 1 · Configuration — fill in your values here

In [ ]:
import base64
import csv as _csv
import json
import math
import subprocess
import time
from pathlib import Path

import requests

# ─────────────────────────────────────────────────────────────────────────────
# ✏️  FILL IN THESE VALUES BEFORE RUNNING
# ─────────────────────────────────────────────────────────────────────────────
GENI_BASE_URL = "https://eag-lumi-backend-532613543802.australia-southeast1.run.app"
GENI_DOMAIN   = "eag"       # e.g. "eag", "genai", "ambiata"
GCLOUD_PATH   = "gcloud"    # full path if gcloud is not on $PATH

# Friendly name that will appear in the GENI document library
DOCUMENT_NAME = "anzsic_master_csv"

# CSV must be in the same folder as this notebook
CSV_PATH = Path(__file__).parent / "anzsic_master.csv" if "__file__" in dir() else Path("anzsic_master.csv")

# Raw bytes per chunk before base64 encoding (256 KB is safe for most proxies)
CHUNK_BYTES = 256 * 1024
# ─────────────────────────────────────────────────────────────────────────────

print(f"CSV path : {CSV_PATH.resolve()}")
print(f"Exists   : {CSV_PATH.exists()}")

## 2 · Authentication helper

In [ ]:
def get_token() -> str:
    """Fetch a fresh GCloud identity token."""
    result = subprocess.run(
        [GCLOUD_PATH, "auth", "print-identity-token"],
        capture_output=True, text=True, check=True, timeout=15,
    )
    return result.stdout.strip()


def make_headers() -> dict:
    return {
        "Authorization": f"Bearer {get_token()}",
        "x-iag-domain":  GENI_DOMAIN,
        "Content-Type":  "application/json",
    }


# Quick auth check
try:
    tok = get_token()
    print(f"✅  Token obtained (first 20 chars): {tok[:20]}...")
except subprocess.CalledProcessError:
    print("❌  gcloud auth failed — run:  gcloud auth login")
    raise

## 3 · Load CSV and inspect

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {CSV_PATH.resolve()}\n"
        "Place anzsic_master.csv in the same folder as this notebook."
    )

raw_bytes = CSV_PATH.read_bytes()
file_size  = len(raw_bytes)

# Quick peek
with CSV_PATH.open(encoding="utf-8") as fh:
    reader  = _csv.DictReader(fh)
    rows    = list(reader)
    columns = reader.fieldnames or []

print(f"File size : {file_size:,} bytes  ({file_size / 1024:.1f} KB)")
print(f"Rows      : {len(rows):,}")
print(f"Columns   : {columns}")
print(f"\nFirst row:")
print(json.dumps(rows[0], indent=2))

## 4 · Split into chunks and base64-encode

The `UploadDocumentChunkRequest` schema requires:
- `upload_key`: `null` on the first chunk, then the UUID returned by the server
- `chunk_idx`: 0-based index
- `total_chunks`: total number of chunks in this upload
- `data`: base64-encoded raw bytes for this chunk
- `length`: total file size in bytes (not chunk size)

In [ ]:
total_chunks = math.ceil(file_size / CHUNK_BYTES)

chunks: list[str] = []
for i in range(total_chunks):
    start = i * CHUNK_BYTES
    end   = min(start + CHUNK_BYTES, file_size)
    chunks.append(base64.b64encode(raw_bytes[start:end]).decode("ascii"))

print(f"CHUNK_BYTES  : {CHUNK_BYTES:,}")
print(f"Total chunks : {total_chunks}")
print(f"Chunk sizes  : {[len(c) for c in chunks]}  (base64 chars)")

## 5 · Upload all chunks to GENI

`POST /api/domainowner/documents` — send one request per chunk.

In [ ]:
UPLOAD_URL = f"{GENI_BASE_URL}/api/domainowner/documents"

upload_key: str | None = None   # null on first chunk; server returns a UUID
bytes_written_total = 0

for idx, chunk_b64 in enumerate(chunks):
    payload = {
        "name":         DOCUMENT_NAME,
        "upload_key":   upload_key,       # null on first request
        "mime_type":    "text/csv",
        "total_chunks": total_chunks,
        "chunk_idx":    idx,
        "length":       file_size,        # total file size in bytes
        "data":         chunk_b64,
    }

    print(f"Uploading chunk {idx + 1}/{total_chunks} ...", end=" ", flush=True)
    resp = requests.post(UPLOAD_URL, headers=make_headers(), json=payload, timeout=120)

    if not resp.ok:
        print(f"\n❌  HTTP {resp.status_code}: {resp.text[:400]}")
        raise RuntimeError(f"Upload failed on chunk {idx}")

    result = resp.json()
    upload_key           = result["upload_key"]   # reuse for all subsequent chunks
    bytes_written        = result["bytes_written"]
    done                 = result["done"]
    bytes_written_total += bytes_written

    print(f"✅  bytes_written={bytes_written:,}  done={done}  key={upload_key}")

print(f"\n🎉  Upload complete. Total bytes written: {bytes_written_total:,}")
print(f"    upload_key (document session ID): {upload_key}")

## 6 · Verify processing and save document IDs

GENI processes the document asynchronously after the last chunk is received (parse → chunk → embed).  
We poll `GET /api/domainowner/documents` until the document appears as active.

In [ ]:
LIST_URL = f"{GENI_BASE_URL}/api/domainowner/documents"

MAX_WAIT_SECS = 300   # 5 min — large CSVs can take a while to embed
POLL_INTERVAL = 10

document_id: str | None = None
print(f"Polling for document '{DOCUMENT_NAME}' to finish processing ...")
deadline = time.monotonic() + MAX_WAIT_SECS

while time.monotonic() < deadline:
    resp = requests.get(LIST_URL, headers=make_headers(), timeout=30)
    if not resp.ok:
        print(f"❌  List call failed: {resp.status_code} {resp.text[:200]}")
        break

    docs  = resp.json().get("documents", [])
    match = next(
        (d for d in docs if d["name"] == DOCUMENT_NAME and not d.get("deleted", True)),
        None,
    )

    if match:
        document_id = match["id"]
        print(f"\n✅  Document is ready!\n")
        print(json.dumps(match, indent=2))
        break

    remaining = int(deadline - time.monotonic())
    print(f"  Not ready yet — retrying in {POLL_INTERVAL}s  (timeout in {remaining}s) ...", end="\r")
    time.sleep(POLL_INTERVAL)

if not document_id:
    # It may still be processing — set document_id manually from the list above if needed
    print("\n⚠️  Document not found after polling. Check the GENI UI or increase MAX_WAIT_SECS.")
    print("    You can still manually note the document ID from the GENI document library.")

In [ ]:
# ── Save artefact for the operator ───────────────────────────────────────────
output = {
    "document_name": DOCUMENT_NAME,
    "document_id":   document_id,   # domain-level document UUID — pass to operator
    "upload_key":    upload_key,    # same value; kept for reference
    "geni_base_url": GENI_BASE_URL,
    "geni_domain":   GENI_DOMAIN,
}

output_path = Path("document_ids.json")
output_path.write_text(json.dumps(output, indent=2))

print("=" * 65)
print("📋  HAND THIS FILE / OUTPUT TO THE OPERATOR:")
print("=" * 65)
print(json.dumps(output, indent=2))
print("=" * 65)
print(f"\n💾  Also saved to: {output_path.resolve()}")
print(
    "\nNote: the operator also needs BOT_ID (the bot's UUID, NOT bot_version_id).\n"
    "They can get it via:\n"
    "  GET /api/bots_v2/<bot-url-suffix>  →  response.id"
)